# BiciTierra Market · Predicció mensual amb separació temporal

Compareu una mitjana de grup, el mateix mes de l’any anterior i regressió lineal regularitzada. El model treballa amb dades sintètiques; no és una previsió de mercat real. Requereix `pandas` i `numpy`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('..') / 'data' / 'originales'
sales = pd.read_csv(DATA / 'sales_history.csv')
sales['month'] = pd.to_datetime(sales['month'], format='%Y-%m')
KEY = ['channel', 'product_line', 'region']
sales = sales.sort_values(KEY + ['month']).reset_index(drop=True)
assert not sales.duplicated(['month'] + KEY).any()
sales.head()

,month,channel,product_line,region,marketing_spend,avg_discount,stock_units,website_sessions,sales_units,revenue,marketing_budget,planned_discount,list_price,campaign,days_in_month,closing_stock_units,stockout
0,2021-01-01,ecommerce,electric,coast,2400.37,0.037,51,5119.0,41,58434.84,2588.44,0.045,1480.00,none,31,10,0
1,2021-02-01,ecommerce,electric,coast,2252.26,0.082,61,5569.0,41,55787.04,2166.71,0.067,1482.20,none,28,20,0
2,2021-03-01,ecommerce,electric,coast,2981.02,0.034,78,6276.0,70,100375.80,3020.43,0.038,1484.41,none,31,8,0
3,2021-04-01,ecommerce,electric,coast,4693.54,NaN,64,7654.0,64,82299.28,4379.29,0.136,1486.62,spring,30,0,1
4,2021-05-01,ecommerce,electric,coast,3415.94,0.026,91,6505.0,69,100058.31,3692.30,0.026,1488.83,none,31,22,0


## 1. Retards i períodes

Ordenem cada sèrie i usem `shift`, mai el valor del mateix mes. El primer any queda fora de l’ajust perquè falta `lag_12`. Entrenament 2022–2024, validació 2025 i prova gener–juny 2026.

In [2]:
# Els retards es calculen dins de cada sèrie i només miren mesos anteriors.
for lag in [1, 12]:
    sales[f'lag_{lag}'] = sales.groupby(KEY)['sales_units'].shift(lag)
usable = sales.dropna(subset=['lag_1', 'lag_12']).copy()
train = usable[usable['month'] < '2025-01-01'].copy()
valid = usable[(usable['month'] >= '2025-01-01') & (usable['month'] < '2026-01-01')].copy()
test = usable[usable['month'] >= '2026-01-01'].copy()
print({'train': len(train), 'valid': len(valid), 'test': len(test)})

{'train': 648, 'valid': 216, 'test': 108}


## 2. Referències i mètriques

La mitjana s’aprén només amb entrenament. MAE és error mitjà en unitats; WAPE és error absolut total respecte al volum real total. Limitem totes les prediccions al rang físic 0–estoc amb la mateixa regla.

In [3]:
def score(y, pred):
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    return {'MAE_units': float(np.mean(np.abs(y-pred))),
            'WAPE_pct': float(100*np.abs(y-pred).sum()/np.abs(y).sum())}

group_mean = train.groupby(KEY)['sales_units'].mean().rename('prediction')
def mean_prediction(frame):
    merged = frame.merge(group_mean.reset_index(), on=KEY, how='left', validate='many_to_one')
    return merged['prediction'].fillna(train['sales_units'].mean()).to_numpy()

def cap(frame, prediction):
    return np.clip(prediction, 0, frame['stock_units'].to_numpy())

### Comprovació de setmana 2

Calculeu les referències amb validació; reserveu la regressió per a setmana 3 i el test per a setmana 5.


In [4]:
reference_results = pd.DataFrame([
    {'model': 'group_mean', **score(valid['sales_units'], cap(valid, mean_prediction(valid)))},
    {'model': 'seasonal_lag12', **score(valid['sales_units'], cap(valid, valid['lag_12']))},
])
reference_results.round(2)


,model,MAE_units,WAPE_pct
0,group_mean,18.75,18.06
1,seasonal_lag12,17.40,16.77


### Semana 2 · Decisiones de preparación

Los CSV de `data/originales/` no se modifican: todas las transformaciones se hacen en memoria.

| Problema | Decisión | Motivo |
|---|---|---|
| Nulos en `marketing_spend`, `avg_discount` y `website_sessions` | Excluir del modelo | Se conocen al cierre del mes; no se pueden usar para predecirlo, así que sus nulos no afectan |
| Nulos en `marketing_budget` (6 en entrenamiento, 3 en prueba) y `planned_discount` (5 en entrenamiento, 7 en validación) | Imputar en el apartado 3 con un valor aprendido solo con entrenamiento | Un nulo no es un cero: rellenar con 0 diría que no hubo presupuesto o descuento |
| 2021 no tiene `lag_12` | Excluir del ajuste; se usa solo para calcular retrasos | Sin el mismo mes del año anterior no se puede comparar con la referencia estacional |
| Censura por stock (`stockout = 1` en el 19 % de los meses) | Conservar las filas y limitar las predicciones entre 0 y `stock_units` | No se puede vender más de lo que hay; la demanda real no atendida no se observa |

### Particiones

Separación temporal, no aleatoria: el modelo nunca ve meses posteriores al que predice.

| Partición | Meses | Filas | Uso |
|---|---|---|---|
| Entrenamiento | enero 2022 – diciembre 2024 | 648 | Aprender medias y, en la semana 3, el modelo |
| Validación | enero 2025 – diciembre 2025 | 216 | Comparar referencias y elegir modelo |
| Prueba | enero 2026 – junio 2026 | 108 | Reservada hasta la semana 5 |

Cada partición tiene 18 series × número de meses (36, 12 y 6).

### Variables admitidas y excluidas

| Admitidas (conocidas antes del mes) | Excluidas (conocidas al cierre) |
|---|---|
| `channel`, `product_line`, `region`, `campaign` | `revenue` |
| `stock_units` (stock inicial) | `website_sessions` |
| `marketing_budget`, `planned_discount`, `list_price` | `marketing_spend`, `avg_discount` |
| `days_in_month`, mes del año | `closing_stock_units`, `stockout` |
| `lag_1`, `lag_12` (ventas de meses ya cerrados) | `sales_units` del mismo mes (es el objetivo) |

- **`revenue`** se excluye porque se conoce el último día del mes y contiene las propias unidades vendidas (≈ unidades × precio × (1 − descuento)).
- **`website_sessions`** se excluye porque se cuenta durante el mes: el 1 de julio no sabemos cuántas visitas tendrá julio.

### Resultados de las referencias en validación (2025)

| Referencia | MAE (unidades) | WAPE |
|---|---|---|
| Media de cada serie en entrenamiento (`group_mean`) | 18,75 | 18,06 % |
| Mismo mes del año anterior (`seasonal_lag12`) | **17,40** | **16,77 %** |

- **`seasonal_lag12` es la referencia a batir.** Recoge la estacionalidad de cada serie (hipótesis 1) y la media plana no.
- **Qué significa para el negocio:** una serie vende unas 104 unidades al mes en 2025, y nos equivocamos de media en unas 17 bicis por serie y mes, un 17 % del volumen. Para el stock significa sobrar o faltar unas 17 bicis en cada canal y región.
- No es comparable con el 19,54 % de la baseline de Andy: allí la media usaba 2021–2024 y no limitaba las predicciones al stock.

### Ejemplo: una variable posterior falsea el resultado

Si usáramos `revenue`, podríamos reconstruir las unidades casi sin error. El resultado parecería perfecto, pero sería inútil: en julio no conocemos el `revenue` de julio hasta que termina el mes.

In [5]:
# Fuga de información: reconstruir las unidades a partir de datos que solo se conocen al cierre
estimacion = valid['revenue'] / (valid['list_price'] * (1 - valid['avg_discount']))
filas = estimacion.notna()
print('Con revenue (fuga):', score(valid.loc[filas, 'sales_units'], estimacion[filas]))
print('Mejor referencia válida:', score(valid['sales_units'], cap(valid, valid['lag_12'])))

Con revenue (fuga): {'MAE_units': 4.156430593019665e-06, 'WAPE_pct': 3.981508399360554e-06}
Mejor referencia válida: {'MAE_units': 17.40277777777778, 'WAPE_pct': 16.76628010704728}


Con `revenue` el error es prácticamente 0, frente a las 17 unidades de la referencia válida. Un error tan bajo es una señal de alarma, no un buen modelo.

**Mala imputación:** algo parecido pasaría si rellenáramos los nulos de `marketing_budget` con 0. El modelo vería meses con presupuesto 0 que en realidad sí tuvieron presupuesto, y aprendería una relación falsa entre marketing y ventas.

> Setmana 3. Continueu ací després de revisar CP2.

## 3. Model lineal

La regularització penalitza coeficients massa grans. Es prova el paràmetre `alpha` en validació. Els predictors exclosos inclouen sessions, ingressos, despesa real, descompte real, estoc final i ruptura del mateix mes. Les variables retardades sí que eren conegudes.

En treball posterior podeu reproduir este procés amb un `Pipeline` de scikit-learn i comparar-lo amb un arbre, mantenint exactament les mateixes dates.

In [6]:
# Regressió lineal regularitzada (ridge) amb NumPy: no necessita scikit-learn.
# Imputació, codificació i escala apreses només amb entrenament.
NUM = ['marketing_budget','planned_discount','stock_units','list_price',
       'days_in_month','lag_1','lag_12']
CAT = KEY + ['campaign']
def design(frame):
    x = frame[NUM].copy()
    month = frame['month'].dt.month
    x['month_sin'] = np.sin(2*np.pi*month/12)
    x['month_cos'] = np.cos(2*np.pi*month/12)
    x['time_index'] = (frame['month'].dt.year-2021)*12+month-1
    for product in ['urban','mountain','electric']:
        x[f'{product}_sin'] = x['month_sin']*(frame['product_line']==product)
        x[f'{product}_cos'] = x['month_cos']*(frame['product_line']==product)
    return pd.concat([x, pd.get_dummies(frame[CAT], dtype=float)], axis=1)

def fit_ridge(frame, alpha):
    x = design(frame)
    columns = x.columns
    medians = x.median()
    x = x.fillna(medians)
    center, scale = x.mean(), x.std().replace(0, 1)
    z = ((x-center)/scale).to_numpy(dtype=float)
    z = np.column_stack([np.ones(len(z)), z])
    penalty = np.eye(z.shape[1])*alpha
    penalty[0,0] = 0 # La constant no es penalitza.
    beta = np.linalg.solve(z.T@z+penalty, z.T@frame['sales_units'].to_numpy())
    return columns, medians, center, scale, beta

def predict_ridge(frame, model):
    columns, medians, center, scale, beta = model
    x = design(frame).reindex(columns=columns, fill_value=0).fillna(medians)
    z = ((x-center)/scale).to_numpy(dtype=float)
    return np.column_stack([np.ones(len(z)), z])@beta

## 4. Comparar en validació

Trieu el model amb 2025. No mireu el test per canviar variables o paràmetres.

In [7]:
validation_results = []
validation_results.append({'model':'group_mean', **score(valid['sales_units'],cap(valid,mean_prediction(valid)))})
validation_results.append({'model':'seasonal_lag12', **score(valid['sales_units'],cap(valid,valid['lag_12']))})
models = {}
for alpha in [1.0, 10.0, 100.0]:
    name = f'ridge_{alpha:g}'
    models[name] = fit_ridge(train, alpha)
    validation_results.append({'model':name, **score(valid['sales_units'],cap(valid,predict_ridge(valid,models[name])))})
comparison = pd.DataFrame(validation_results).sort_values('MAE_units').reset_index(drop=True)
chosen = comparison.iloc[0]['model']
print(comparison.round(2).to_string(index=False))
print('Model seleccionat només amb validació:', chosen)

         model  MAE_units  WAPE_pct
     ridge_100      12.05     11.61
      ridge_10      12.37     11.92
       ridge_1      12.43     11.98
seasonal_lag12      17.40     16.77
    group_mean      18.75     18.06
Model seleccionat només amb validació: ridge_100


### Semana 3 · Qué aporta el modelo lineal regularizado

Las referencias usan una sola pista: la media de la serie o el mismo mes del año anterior. El modelo ridge combina a la vez todas las variables conocidas antes del mes: retrasos, stock inicial, presupuesto y descuento previstos, precio, campaña, estacionalidad por producto y tendencia (`time_index`).

La **regularización** (`alpha`) castiga los coeficientes grandes. Así el modelo no se ajusta de más a los años de entrenamiento. La imputación (medianas), la codificación y el escalado se aprenden **solo con entrenamiento** y se aplican igual a validación.

### Comparativa en validación (2025)

| Modelo | Variables | Parámetro | MAE (unidades) | WAPE |
|---|---|---|---|---|
| `group_mean` | Media de la serie | — | 18,75 | 18,06 % |
| `seasonal_lag12` | Mismo mes del año anterior | — | 17,40 | 16,77 % |
| `ridge_1` | Todas las admitidas | `alpha = 1` | 12,43 | 11,98 % |
| `ridge_10` | Todas las admitidas | `alpha = 10` | 12,37 | 11,92 % |
| **`ridge_100`** | Todas las admitidas | **`alpha = 100`** | **12,05** | **11,61 %** |

Todos se evalúan con las mismas 216 filas de 2025, la misma regla de recorte entre 0 y stock y las mismas métricas.

### Elección provisional: `ridge_100`

- Baja el error de 17,40 a 12,05 unidades por serie y mes (un 31 % menos que la mejor referencia). La mejora compensa la complejidad: son unas 5 bicis menos de error en cada canal y región cada mes.
- Las tres `alpha` dan resultados muy parecidos (12,05–12,43), así que la elección no depende de un ajuste fino.
- En entrenamiento el error es de 9,93 unidades y en validación de 12,05: empeora un poco con datos nuevos, pero no hay un sobreajuste grave.
- Las variables con más peso son `stock_units`, `lag_12` y `lag_1`. Encaja con lo visto: el stock limita las ventas y la estacionalidad importa.

La elección se revisa en el CP3. El test de 2026 sigue sin abrirse.

### Casos de error en validación


In [8]:
# Errores mayores del modelo elegido en validación (2025), comparados con la referencia
casos = valid[['month'] + KEY + ['campaign', 'stock_units', 'stockout', 'lag_12', 'sales_units']].copy()
casos['pred_ridge'] = cap(valid, predict_ridge(valid, models[chosen])).round(1)
casos['error'] = (casos['pred_ridge'] - casos['sales_units']).round(1)
casos.reindex(casos['error'].abs().sort_values(ascending=False).index).head(5)

,month,channel,product_line,region,campaign,stock_units,stockout,lag_12,sales_units,pred_ridge,error
449,2025-06-01,ecommerce,urban,coast,none,237,1,146.0,237,169.3,-67.7
383,2025-06-01,ecommerce,mountain,metro,none,249,0,145.0,228,160.9,-67.1
579,2025-04-01,ecommerce,urban,metro,none,236,0,189.0,234,177.7,-56.3
581,2025-06-01,ecommerce,urban,metro,none,235,0,197.0,224,172.7,-51.3
381,2025-04-01,ecommerce,mountain,metro,spring,210,1,162.0,210,160.9,-49.1


**Caso 1 · junio 2025, `ecommerce` + `urban` + `coast`.** El modelo predice 169 unidades y se venden 237, justo todo el stock (`stockout = 1`). La demanda real fue **al menos** 237 y no sabemos cuánto más. El modelo falla por abajo porque aprende de meses en los que las ventas ya estaban limitadas por el stock. En los meses con rotura, el error medio es siempre por defecto (−11,4 unidades).

**Caso 2 · junio 2025, `ecommerce` + `mountain` + `metro`.** Sin rotura de stock: se venden 228 y el modelo predice 161 (`lag_12` decía 145). La serie ha crecido mucho respecto a 2024 y el modelo no lo recoge del todo. Encaja con la hipótesis 2: el canal `ecommerce` crece y la tendencia reciente pesa más que la media de años anteriores.

**Patrón:** los errores mayores se concentran en `ecommerce`, en `metro` y en `urban`, y casi siempre son por defecto (el modelo se queda corto). Para el stock, eso significa riesgo de quedarse sin bicis en las series que más venden.


> Setmana 5. Atureu-vos ací fins que el model estiga triat i revisat en CP3. No useu el test per afinar decisions.

## 5. Prova final

Fixada la decisió anterior, avaluem 2026 una sola vegada. L’ajust queda congelat; els retards s’actualitzen amb mesos que ja s’han observat. És predicció mensual successiva, no una predicció simultània de sis mesos.

**Decisión congelada antes de abrir la prueba (CP3):** modelo `ridge_100`; predictores `marketing_budget`, `planned_discount`, `stock_units`, `list_price`, `days_in_month`, `lag_1`, `lag_12`, mes del año (seno y coseno, también por producto), `time_index`, canal, producto, región y campaña; `alpha = 100`; imputación, codificación y escala aprendidas con entrenamiento 2022–2024; predicciones limitadas entre 0 y `stock_units`.

In [9]:
def selected_prediction(frame):
    if chosen == 'group_mean': return cap(frame,mean_prediction(frame))
    if chosen == 'seasonal_lag12': return cap(frame,frame['lag_12'])
    return cap(frame,predict_ridge(frame,models[chosen]))

# Prova d'un mes vista: el retard 1 de juny usa maig, ja observat en aquell moment.
test_prediction = selected_prediction(test)
print('Prova final:', score(test['sales_units'], test_prediction))
errors = test[['month']+KEY+['sales_units']].copy()
errors['prediction'] = test_prediction
errors['abs_error'] = abs(errors['sales_units']-errors['prediction'])
errors.sort_values('abs_error',ascending=False).head(10)

Prova final: {'MAE_units': 16.135532745993526, 'WAPE_pct': 13.531895764616406}


,month,channel,product_line,region,sales_units,prediction,abs_error
591,2026-04-01,ecommerce,urban,metro,328,229.920076,98.079924
391,2026-02-01,ecommerce,mountain,metro,158,114.755912,43.244088
1182,2026-01-01,store,urban,metro,185,142.551806,42.448194
196,2026-05-01,ecommerce,electric,metro,145,103.822016,41.177984
590,2026-03-01,ecommerce,urban,metro,230,189.347564,40.652436
1053,2026-04-01,store,urban,coast,210,169.435658,40.564342
1187,2026-06-01,store,urban,metro,202,162.677511,39.322489
1185,2026-04-01,store,urban,metro,220,181.546382,38.453618
63,2026-04-01,ecommerce,electric,coast,143,107.465570,35.534430
527,2026-06-01,ecommerce,urban,inland,167,133.935788,33.064212


In [10]:
# Contexto (no sirve para elegir): las referencias en el mismo periodo de prueba
prueba = test.copy()
prueba['prediccion'] = test_prediction
prueba['error'] = prueba['prediccion'] - prueba['sales_units']
resumen_prueba = pd.DataFrame([
    {'model': chosen, **score(test['sales_units'], test_prediction)},
    {'model': 'seasonal_lag12', **score(test['sales_units'], cap(test, test['lag_12']))},
    {'model': 'group_mean', **score(test['sales_units'], cap(test, mean_prediction(test)))},
])
print(resumen_prueba.round(2).to_string(index=False))
print('Error medio con signo (negativo = nos quedamos cortos):', round(prueba['error'].mean(), 2))
print('Venta media por serie y mes en la prueba:', round(prueba['sales_units'].mean(), 1), '| en validación:', round(valid['sales_units'].mean(), 1))
pd.concat([comparison.assign(periodo='validación 2025'), resumen_prueba.assign(periodo='prueba ene–jun 2026')]).round(2).to_csv('../evidencies/metricas_modelos.csv', index=False)

         model  MAE_units  WAPE_pct
     ridge_100      16.14     13.53
seasonal_lag12      21.24     17.81
    group_mean      26.93     22.58
Error medio con signo (negativo = nos quedamos cortos): -14.34
Venta media por serie y mes en la prueba: 119.2 | en validación: 103.8


### Semana 5 · Resultado de la prueba

| | MAE (unidades) | WAPE |
|---|---|---|
| `ridge_100` en validación 2025 | 12,05 | 11,61 % |
| **`ridge_100` en prueba ene–jun 2026** | **16,14** | **13,53 %** |
| `seasonal_lag12` en prueba (solo contexto) | 21,24 | 17,81 % |

- **El error sube con datos nuevos**, de 12,05 a 16,14 unidades. Es lo normal: la validación sirvió para elegir y la prueba mide lo que pasa con meses que el modelo nunca ha visto. En porcentaje sube menos (de 11,6 % a 13,5 %) porque en 2026 se vende más.
- El modelo sigue mejorando a la referencia estacional en el mismo periodo. Este dato no se ha usado para elegir nada.
- **Sesgo:** el error medio con signo es −14,3 unidades. El modelo se queda corto de forma sistemática, porque en 2026 la venta media por serie sube a 119 unidades frente a 104 en 2025 y el modelo no sigue del todo ese crecimiento.
- El peor mes es abril (24,5 unidades de error), con la campaña `spring`. La región `metro` es la que más falla (21,7 unidades).

### Casos de error en la prueba

**Caso 1 · abril 2026, `ecommerce` + `urban` + `metro`, campaña `spring`.** Se venden 328 bicis y el modelo predice 230 (98 de error). No hubo rotura de stock: había 356 bicis. El año anterior se vendieron 234 y el mes anterior 230; el salto en primavera es mayor que nunca y el modelo, que se apoya en esos retrasos, no lo anticipa.

**Caso 2 · enero 2026, `store` + `urban` + `metro`.** Se venden 185 y el modelo predice 143. En un mes sin campaña, la serie vende un 54 % más que en enero de 2025 (120). Es otra vez crecimiento que el modelo no capta.

**Rotura de stock:** en los 21 meses de prueba con `stockout = 1`, el error es siempre por defecto. Como en validación, la demanda real fue mayor de lo que vemos.

### Qué influye en el modelo y límites

Las variables con más peso son `stock_units`, `lag_12` y `lag_1`. Que el stock pese tanto no significa que poner más stock cree ventas: refleja que las ventas están limitadas por el stock y que la empresa ya pone más stock donde espera vender más. Con datos sintéticos, una relación aprendida no demuestra una causa en el mercado real.

## 6. Aplicar a juliol de 2026

No hi ha objectiu real per comparar. Usem el mateix model triat i congelat per fer el procés fàcil de seguir. Reentrenar amb tot l’històric abans de producció seria una ampliació posterior; caldria distingir-ne les mètriques de les de la prova.

In [11]:
future = pd.read_csv(DATA / 'sales_forecast_input.csv')
future['month'] = pd.to_datetime(future['month'],format='%Y-%m')
# Correspondència exacta de mes i sèrie, sense usar files d'altres grups.
for lag in [1,12]:
    past = sales[['month']+KEY+['sales_units']].copy()
    past['month'] = past['month'] + pd.DateOffset(months=lag)
    past = past.rename(columns={'sales_units':f'lag_{lag}'})
    future = future.merge(past,on=['month']+KEY,how='left',validate='one_to_one')
assert future[['lag_1','lag_12']].notna().all().all()
forecast = future[['month']+KEY].copy()
forecast['predicted_sales_units'] = np.rint(selected_prediction(future)).astype(int)
# Se guarda como previsión: julio todavía no tiene venta real para comparar
forecast.to_csv('../evidencies/prediccions_juliol_2026.csv', index=False)
forecast

,month,channel,product_line,region,predicted_sales_units
0,2026-07-01,store,urban,coast,172
1,2026-07-01,store,urban,inland,112
2,2026-07-01,store,urban,metro,198
3,2026-07-01,store,mountain,coast,135
4,2026-07-01,store,mountain,inland,111
5,2026-07-01,store,mountain,metro,108
6,2026-07-01,store,electric,coast,92
7,2026-07-01,store,electric,inland,45
8,2026-07-01,store,electric,metro,101
9,2026-07-01,ecommerce,urban,coast,224


In [12]:
# Series en las que la predicción toca el stock disponible
julio = future[KEY + ['campaign', 'stock_units']].copy()
julio['prediccion'] = forecast['predicted_sales_units']
julio['uso_stock'] = (julio['prediccion'] / julio['stock_units']).round(2)
print('Unidades previstas:', julio['prediccion'].sum(), 'de', julio['stock_units'].sum(), 'en stock')
julio.sort_values('uso_stock', ascending=False).head(6)

Unidades previstas: 2398 de 2951 en stock


,channel,product_line,region,campaign,stock_units,prediccion,uso_stock
1,store,urban,inland,summer,112,112,1.00
5,store,mountain,metro,summer,108,108,1.00
7,store,electric,inland,summer,45,45,1.00
3,store,mountain,coast,summer,136,135,0.99
10,ecommerce,urban,inland,summer,159,154,0.97
13,ecommerce,mountain,inland,summer,117,108,0.92


### Previsión de julio de 2026 (no validada)

Se guardan las 18 predicciones en `evidencies/prediccions_juliol_2026.csv`, con la clave completa. **Es una previsión, no un resultado:** la venta real de julio no está en los datos.

- Total previsto: 2 398 bicis, con 2 951 en stock. En julio de 2025 se vendieron 2 432.
- En tres series de tienda física la predicción llega al stock disponible (`urban` + `inland`, `mountain` + `metro` y `electric` + `inland`). Ahí la venta podría quedar limitada por el stock.

### Recomendaciones

**Stock (confianza media).** Subir el stock de julio en las tres series de tienda física que tocan el límite. Además, como el modelo se queda corto de media en unas 14 unidades por serie en 2026, conviene añadir un margen de seguridad de ese orden en `ecommerce` + `metro` y en `urban`, que es donde más se queda corto. Habría que comprobarlo con la venta real de julio antes de convertirlo en una regla.

**Campaña (confianza baja).** La campaña `summer` coincide con el mayor pico de ventas, pero coincide también con la temporada, así que no podemos separar el efecto de la campaña del de la estación (hipótesis 1). Recomendamos mantenerla y medir su efecto con una prueba: por ejemplo, variar el presupuesto en algunas regiones y comparar.

## Entrega

Expliqueu el període de cada partició, la diferència entre predictors planificats i observats, el model triat i el seu error final. Mostreu un cas on falle. No presenteu la previsió de juliol com a resultat validat: la resposta encara no està disponible en el cas.

### Respuesta a la entrega

- **Particiones:** entrenamiento enero 2022 – diciembre 2024 (648 filas); validación 2025 (216); prueba enero – junio 2026 (108). 2021 solo sirve para calcular `lag_12`.
- **Predictores planificados y observados:** los planificados se fijan antes del mes (`marketing_budget`, `planned_discount`, `list_price`, `stock_units`, `campaign`) y se pueden usar. Los observados se conocen al cierre (`marketing_spend`, `avg_discount`, `revenue`, `website_sessions`, `stockout`) y no se pueden usar para predecir ese mismo mes.
- **Modelo elegido:** `ridge_100`, elegido solo con validación (MAE 12,05; WAPE 11,61 %).
- **Error final en la prueba:** MAE 16,14 unidades; WAPE 13,53 %.
- **Un caso en que falla:** abril 2026, `ecommerce` + `urban` + `metro`, predice 230 y se venden 328.
- **Julio:** la previsión no está validada; no hay venta real para comparar.